In [ ]:
import os
import sys
import torch
from pathlib import Path
from hydra import initialize_config_dir, compose


PROJECT_ROOT = Path.cwd().parent
if PROJECT_ROOT not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from tracers.datasets import (
    AutoOsu4KDataset,
    split_song_ids,
    make_dataloader,
)

ROOT = Path(os.environ.get("AUTOOSU_DATASET_DIR", PROJECT_ROOT / "dataset")).expanduser().resolve()

AUDIO_DIR = ROOT / "audio"
BEATMAP_DIR = ROOT / "beatmap" / "4keys"
CONFIG_DIR = PROJECT_ROOT / "conf"


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

with initialize_config_dir(
    version_base=None,
    config_dir=str(CONFIG_DIR),
):
    cfg = compose(config_name="train")


experiment_config = cfg.experiment
hyperparams = cfg.hyperparams
experiment_config.checkpoint_path = str((PROJECT_ROOT / experiment_config.checkpoint_path).resolve())

segment_frames = hyperparams["segment_frames"]
batch_size = hyperparams["batch_size"]



In [ ]:

base_dataset = AutoOsu4KDataset(
    beatmap_dir=BEATMAP_DIR,
    audio_dir=AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="random",
    audio_cache_size=8,
)

train_ids, val_ids, test_ids = split_song_ids(
    base_dataset.song_ids,
    val_fraction=0.10,
    test_fraction=0.10,
    seed=42,
)



In [ ]:


train_dataset = AutoOsu4KDataset(
    BEATMAP_DIR,
    AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="random",
    crops_per_chart=2,
    song_ids=train_ids,
)

val_dataset = AutoOsu4KDataset(
    BEATMAP_DIR,
    AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="all",
    song_ids=val_ids,
)

test_dataset = AutoOsu4KDataset(
    BEATMAP_DIR,
    AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="all",
    song_ids=test_ids,
)



In [ ]:
train_loader = make_dataloader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

val_loader = make_dataloader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

test_loader = make_dataloader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

In [ ]:
from tracers.models.transformer import TracerNpModel

model = TracerNpModel(hyperparams).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hyperparams["learning_rate"],
    weight_decay=hyperparams["weight_decay"]
)

In [ ]:
from tracers.training.trainer import Trainer

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    valid_loader=val_loader,
    device=device,
    experiment_config=experiment_config,
    hyperparams=hyperparams,
)

# Load checkpoint if specified in YAML
if experiment_config.resume_from is not None:
    resume_from = PROJECT_ROOT / experiment_config.resume_from
    trainer.load_checkpoint(resume_from)
    for g in trainer.optimizer.param_groups:
        g["lr"] = hyperparams["learning_rate"]

    print("learning_rate set to ", hyperparams["learning_rate"])


In [ ]:
trainer.fit()